# Parametric PINN simulation (phi1, phi2 only)

This notebook configures `ParametricPIPNNs` so only `phi1` and `phi2` are learned, while `mx`, `my`, and `k` stay fixed.


In [ ]:
import numpy as np
from pinn_ndof_chain_parametric_tf2 import lhs_sample, ParametricPIPNNs

# --- fixed structural parameters ---
fixed_params = {'mx': 1.0, 'my': 0.3, 'k': 1.0}

# --- only sweep forcing parameters ---
param_names = ['phi1', 'phi2']
lb_params = np.array([1.0, 10.0], dtype=np.float32)
ub_params = np.array([2.0, 20.0], dtype=np.float32)
n_cases = 20
params_cases = lhs_sample(n_cases, lb_params, ub_params, seed=42)

# --- model/global settings ---
n_dof = 20
T_seg = 1.0
N_col = 1000
D = 1.0
c_damp = 0.0
layers = [1 + 5, 64, 64, n_dof]  # input stays [t,mx,my,k,phi1,phi2]
hyp_ini_weight_loss = [1.0, 1.0]

# --- initial conditions (per case) ---
phi_offsets = np.zeros((n_cases,), dtype=np.float32)
x0_cases = np.zeros((n_cases, n_dof), dtype=np.float32)
xt0_cases = np.zeros((n_cases, n_dof), dtype=np.float32)
y0_cases = np.zeros((n_cases, n_dof), dtype=np.float32)
yt0_cases = np.zeros((n_cases, n_dof), dtype=np.float32)

model = ParametricPIPNNs(
    n_dof=n_dof,
    params_cases=params_cases,
    lb_params=lb_params,
    ub_params=ub_params,
    c_damp=c_damp,
    T_seg=T_seg,
    N_col=N_col,
    phi_offsets=phi_offsets,
    x0_cases=x0_cases,
    xt0_cases=xt0_cases,
    y0_cases=y0_cases,
    yt0_cases=yt0_cases,
    D=D,
    layers=layers,
    hyp_ini_weight_loss=hyp_ini_weight_loss,
    optimizer_LB=True,
    fixed_params=fixed_params,
    param_names=param_names,
)

# Train (optional)
# model.train(nIter=1000, optimizer_LB=True)
